<a href="https://www.kaggle.com/code/kedhareswernaidu/moonknight?scriptVersionId=231369566" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Like Paintings

### Environment Setup and Imports

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, initializers, optimizers
import numpy as np
import matplotlib.pyplot as plt

## --------------------------
## 1. Custom Instance Normalization
## (Replaces TF-Addons' InstanceNormalization)
## --------------------------
class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(InstanceNormalization, self).__init__()
        self.epsilon = epsilon

    def build(self, input_shape):
        self.scale = self.add_weight(
            name='scale',
            shape=(input_shape[-1],),
            initializer='ones')
        self.offset = self.add_weight(
            name='offset',
            shape=(input_shape[-1],),
            initializer='zeros')

    def call(self, x):
        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        inv = tf.math.rsqrt(variance + self.epsilon)
        normalized = (x - mean) * inv
        return self.scale * normalized + self.offset

## --------------------------
## 2. Residual Block
## --------------------------
def residual_block(x, filters):
    """Modified residual block without TF-Addons dependencies"""
    init = x
    x = layers.Conv2D(filters, 3, padding='same', 
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    return layers.Add()([init, x])

## --------------------------
## 3. Generator Architecture
## --------------------------
def build_generator():
    inputs = layers.Input(shape=[256, 256, 3])
    
    # Downsampling
    x = layers.Conv2D(64, 4, strides=2, padding='same', 
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(inputs)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(128, 4, strides=2, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(256, 4, strides=2, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    
    # Residual blocks
    for _ in range(6):
        x = residual_block(x, 256)
    
    # Upsampling
    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same',
                              kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same',
                             kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    
    # Final layer
    x = layers.Conv2DTranspose(3, 4, strides=1, padding='same',
                             kernel_initializer=initializers.RandomNormal(0, 0.02),
                             activation='tanh')(x)
    
    return models.Model(inputs=inputs, outputs=x)

## --------------------------
## 4. PatchGAN Discriminator
## --------------------------
def build_discriminator():
    inputs = layers.Input(shape=[256, 256, 3])
    
    x = layers.Conv2D(64, 4, strides=2, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(inputs)
    x = layers.LeakyReLU(0.2)(x)
    
    x = layers.Conv2D(128, 4, strides=2, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    
    x = layers.Conv2D(256, 4, strides=2, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    
    x = layers.Conv2D(512, 4, strides=1, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    
    x = layers.Conv2D(1, 4, strides=1, padding='same',
                     kernel_initializer=initializers.RandomNormal(0, 0.02))(x)
    
    return models.Model(inputs=inputs, outputs=x)

## --------------------------
## 5. Initialize Models
## --------------------------
generator_G = build_generator()  # Photo → Monet
generator_F = build_generator()  # Monet → Photo
discriminator_X = build_discriminator()  # Monet discriminator
discriminator_Y = build_discriminator()  # Photo discriminator

## --------------------------
## 6. Loss Functions
## --------------------------
def discriminator_loss(real, fake):
    real_loss = tf.reduce_mean(real)
    fake_loss = tf.reduce_mean(fake)
    return fake_loss - real_loss  # Wasserstein loss

def generator_loss(fake):
    return -tf.reduce_mean(fake)  # Wasserstein loss

## --------------------------
## 7. Optimizers
## --------------------------
generator_optimizer = optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

print("Core architecture setup complete!")

Core architecture setup complete!


In [2]:
## --------------------------
## 1. Gradient Penalty (WGAN-GP)
## Replaces TF-Addons' gradient penalty
## --------------------------
def gradient_penalty(discriminator, real_images, fake_images, batch_size):
    """Calculates gradient penalty for WGAN-GP"""
    # Random weight term for interpolation
    alpha = tf.random.uniform(shape=[batch_size, 1, 1, 1], minval=0., maxval=1.)
    interpolated = real_images + alpha * (fake_images - real_images)

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        pred = discriminator(interpolated, training=True)
    
    grads = tape.gradient(pred, [interpolated])[0]
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
    gp = tf.reduce_mean((norm - 1.0)**2)
    return gp

## --------------------------
## 2. Training Step Function
## --------------------------
@tf.function
def train_step(real_x, real_y):
    batch_size = tf.shape(real_x)[0]
    
    with tf.GradientTape(persistent=True) as tape:
        # Forward pass
        fake_y = generator_G(real_x, training=True)
        cycled_x = generator_F(fake_y, training=True)
        
        # Backward pass
        fake_x = generator_F(real_y, training=True)
        cycled_y = generator_G(fake_x, training=True)
        
        # Identity mapping
        same_y = generator_G(real_y, training=True)
        same_x = generator_F(real_x, training=True)
        
        # Discriminator outputs
        disc_real_x = discriminator_X(real_x, training=True)
        disc_fake_x = discriminator_X(fake_x, training=True)
        
        disc_real_y = discriminator_Y(real_y, training=True)
        disc_fake_y = discriminator_Y(fake_y, training=True)
        
        # Generator losses
        gen_G_loss = generator_loss(disc_fake_y)
        gen_F_loss = generator_loss(disc_fake_x)
        
        # Cycle consistency loss
        cycle_loss = tf.reduce_mean(tf.abs(real_x - cycled_x)) + \
                    tf.reduce_mean(tf.abs(real_y - cycled_y))
        
        # Identity loss
        identity_loss = tf.reduce_mean(tf.abs(real_y - same_y)) + \
                       tf.reduce_mean(tf.abs(real_x - same_x))
        
        # Total generator loss (with weights)
        total_gen_G_loss = gen_G_loss + 10.0 * cycle_loss + 5.0 * identity_loss
        total_gen_F_loss = gen_F_loss + 10.0 * cycle_loss + 5.0 * identity_loss
        
        # Discriminator losses
        disc_X_loss = discriminator_loss(disc_real_x, disc_fake_x)
        disc_Y_loss = discriminator_loss(disc_real_y, disc_fake_y)
        
        # Add gradient penalty
        gp_X = gradient_penalty(discriminator_X, real_x, fake_x, batch_size)
        gp_Y = gradient_penalty(discriminator_Y, real_y, fake_y, batch_size)
        
        disc_X_loss += 10.0 * gp_X
        disc_Y_loss += 10.0 * gp_Y
    
    # Calculate gradients
    generator_G_gradients = tape.gradient(
        total_gen_G_loss, generator_G.trainable_variables)
    generator_F_gradients = tape.gradient(
        total_gen_F_loss, generator_F.trainable_variables)
    
    discriminator_X_gradients = tape.gradient(
        disc_X_loss, discriminator_X.trainable_variables)
    discriminator_Y_gradients = tape.gradient(
        disc_Y_loss, discriminator_Y.trainable_variables)
    
    # Apply gradients
    generator_optimizer.apply_gradients(
        zip(generator_G_gradients, generator_G.trainable_variables))
    generator_optimizer.apply_gradients(
        zip(generator_F_gradients, generator_F.trainable_variables))
    
    discriminator_optimizer.apply_gradients(
        zip(discriminator_X_gradients, discriminator_X.trainable_variables))
    discriminator_optimizer.apply_gradients(
        zip(discriminator_Y_gradients, discriminator_Y.trainable_variables))
    
    return {
        'gen_G_loss': total_gen_G_loss,
        'gen_F_loss': total_gen_F_loss,
        'disc_X_loss': disc_X_loss,
        'disc_Y_loss': disc_Y_loss,
        'cycle_loss': cycle_loss,
        'identity_loss': identity_loss
    }

## --------------------------
## 3. Sample Generation Utility
## --------------------------
def generate_samples(generator, input_dataset, num_samples=1):
    for sample in input_dataset.take(num_samples):
        prediction = generator(sample, training=False)
        plt.figure(figsize=(10, 5))
        
        plt.subplot(1, 2, 1)
        plt.title("Input Image")
        plt.imshow(sample[0] * 0.5 + 0.5)  # Scale from [-1,1] to [0,1]
        
        plt.subplot(1, 2, 2)
        plt.title("Generated Image")
        plt.imshow(prediction[0] * 0.5 + 0.5)
        plt.show()

## --------------------------
## 4. Training Loop
## --------------------------
def train(dataset, epochs, save_interval=5):
    for epoch in range(epochs):
        epoch_losses = {
            'gen_G_loss': [],
            'gen_F_loss': [],
            'disc_X_loss': [],
            'disc_Y_loss': [],
            'cycle_loss': [],
            'identity_loss': []
        }
        
        for image_x, image_y in dataset:
            losses = train_step(image_x, image_y)
            for key in epoch_losses:
                epoch_losses[key].append(losses[key])
        
        # Print epoch statistics
        print(f"\nEpoch {epoch + 1}/{epochs}")
        for key, values in epoch_losses.items():
            print(f"{key}: {np.mean(values):.4f}", end=' | ')
        
        # Generate samples periodically
        if (epoch + 1) % save_interval == 0:
            print("\nGenerating samples...")
            generate_samples(generator_G, photo_ds, num_samples=2)
            generate_samples(generator_F, monet_ds, num_samples=2)

print("Training components setup complete!")

Training components setup complete!
